# 1 - Build `model_df`

**Stage 1 of 4** in the cross-sectional equity alpha pipeline (split out of the old monolithic `alpha_backtest.ipynb`).

This notebook assembles the point-in-time modelling panel end to end:

1. Point-in-time S&P 500 universe + CIK entity keys (survivorship-bias fix)
2. Price download (Yahoo) and PIT membership filter
3. Equity features (momentum, vol, Yang-Zhang, MA-spread, Hurst, ADV)
4. FRED macro + ETF regime features
5. FINRA short volume / interest and SEC fundamentals
6. Strict-lag master assembly, market-factor residualization, winsorise + z-score

**Requires** `FRED_API_KEY` in the environment and the cached parquet in `Data/` (see `Data/README.md`). The Yahoo/SEC pulls are slow, so this is a run-once stage.

**Output** &rarr; `Data/interim/model_df.parquet` and `Data/interim/pipeline_meta.json`, consumed by notebook **2 - walk-forward & models**.


In [ ]:
# =============================================================
# Cell 0 — Dependencies
# =============================================================
# Everything needed is pinned in requirements.txt at the repo root.
# Uncomment on a fresh environment (e.g. Colab):

# %pip install -r ../requirements.txt --quiet

In [1]:
# =============================================================
# Cell 1 — Imports & Global Settings
# =============================================================
# All imports and global constants defined upfront.
# START_DATE covers our full backtest window.
# =============================================================

import os
import pandas as pd
import numpy as np
import requests
import warnings
import yfinance as yf
import cvxpy as cp

from io import StringIO
from tqdm import tqdm
from fredapi import Fred
from scipy.stats import spearmanr
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

# --- Global constants ---
START_DATE   = "2020-01-01"

# v18: prediction horizon in trading days. The target (Cell 11) is the
# per-date DEMEANED forward return over this horizon. Chosen a priori:
# a 1-day horizon implied ~114x annual turnover in v17, which no
# realistic cost assumption survives; ~1 week targets slower signals.
TARGET_HORIZON = 5

# v19: portfolios rebalance every TARGET_HORIZON trading days and earn
# the H-day PERIOD return, so a year contains 252/H observations —
# annualisation MUST use this factor, not 252. (The v18 run annualised
# 5-day period returns with √252, inflating every Sharpe by √5 ≈ 2.24
# and every annualised vol by the same factor.)
REBALANCE_EVERY  = TARGET_HORIZON
PERIODS_PER_YEAR = 252 / REBALANCE_EVERY
FRED_API_KEY = os.getenv("FRED_API_KEY")  # free key from fred.stlouisfed.org

# --- File paths (relative to the repo root; see Data/README.md) ---
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
PATH_SP500_HISTORY = "Data/S&P 500 Historical Components & Changes (Updated).csv"
PATH_SHORT_VOL     = "Data/short_vol.parquet"
PATH_SHORT_INT     = "Data/short_int.parquet"
PATH_FUNDAMENTALS  = "Data/fundamentals.parquet"

print("Cell 1: Imports complete (including statsmodels for residualization).")
print(f"Cell 1: Data start date: {START_DATE}")

: 

In [ ]:
# =============================================================
# Cell 2 — Point-in-Time Universe Construction
# =============================================================
# KEY UPGRADE FROM V10:
# V10 used today's S&P 500 constituent list applied to all
# of history. This is SURVIVORSHIP BIAS — stocks that were
# removed from the index (often poor performers) are excluded,
# making the backtest look artificially good.
#
# V11 FIX:
# We use a historical constituent file that records exactly
# which tickers were in the S&P 500 on each change date.
# The file only records rows when the composition CHANGED,
# so we forward-fill to get the constituent set for every
# trading day.
#
# HOW IT WORKS:
# 1. Load the change file (one row per composition change)
# 2. Build a complete daily calendar from START_DATE to today
# 3. Forward-fill: each day inherits the most recent known
#    constituent list
# 4. For any given date, we look up which tickers were valid
#    and ONLY use those in our universe that day
#
# SYMBOLOGY NOTE:
# The history file uses dots (BRK.B, BF.B).
# Yahoo Finance and our other data sources use dashes (BRK-B).
# We normalise to dashes throughout.
#
# RESULT: a dictionary {date -> set of valid tickers}
# that we use to filter our panel at every step.
# =============================================================

print("Cell 2: Loading point-in-time S&P 500 constituent history...")

sp_hist = pd.read_csv(PATH_SP500_HISTORY)
sp_hist["date"] = pd.to_datetime(sp_hist["date"]).astype("datetime64[ns]")
sp_hist = sp_hist.sort_values("date").reset_index(drop=True)

print(f"Cell 2: History file rows: {len(sp_hist)}")
print(f"Cell 2: Date range: {sp_hist['date'].min().date()} → {sp_hist['date'].max().date()}")

# Build a full business-day calendar from START_DATE to today
bday_calendar = pd.bdate_range(start=START_DATE, end=pd.Timestamp.today()).astype("datetime64[ns]")

# Forward-fill constituent lists to every business day
# merge_asof finds the most recent change on or before each calendar date
sp_hist_clean = sp_hist[sp_hist["date"] <= bday_calendar[-1]].copy()

daily_constituents = pd.merge_asof(
    pd.DataFrame({"date": bday_calendar}),
    sp_hist_clean[["date", "tickers"]],
    on="date",
    direction="backward"   # use the most recent constituent list
)

# Parse comma-separated ticker strings → sets, normalise dots to dashes
def parse_tickers(ticker_str):
    """Split comma-separated ticker string and normalise dot → dash."""
    if pd.isna(ticker_str):
        return set()
    return set(t.strip().replace(".", "-") for t in ticker_str.split(","))

daily_constituents["ticker_set"] = daily_constituents["tickers"].apply(parse_tickers)

# Build lookup dictionary: date → set of valid tickers
# This is our point-in-time universe filter
pit_universe = dict(zip(
    daily_constituents["date"],
    daily_constituents["ticker_set"]
))

# Get the union of all tickers ever in the index since START_DATE
# (used to know which tickers to download price data for)
all_tickers_ever = set()
for ts in daily_constituents["ticker_set"]:
    all_tickers_ever.update(ts)

all_tickers_ever = sorted(all_tickers_ever)

print(f"Cell 2: Business days in calendar: {len(bday_calendar)}")
print(f"Cell 2: Total unique tickers ever in index since {START_DATE}: {len(all_tickers_ever)}")
print(f"Cell 2: Sample tickers for {bday_calendar[0].date()}: {sorted(list(pit_universe[bday_calendar[0]]))[:10]}")
print("Cell 2: Point-in-time universe ready — survivorship bias eliminated.")

In [ ]:
# =============================================================
# Cell 2b — CIK Entity Mapping (NEW in v17)
# =============================================================
# WHY CIK INSTEAD OF TICKER:
# Ticker symbols change over time (FB→META, ANTM→ELV,
# DISCA→WBD, ...). If the entity key is the raw ticker, a
# renamed company is silently treated as TWO different stocks:
# its feature history breaks at the rename date and joins
# against short-volume / fundamentals data can miss.
# The SEC Central Index Key (CIK) is a permanent company
# identifier that never changes, so we use it as the stable
# entity key throughout the pipeline.
#
# ONE SUBTLETY — MULTIPLE SHARE CLASSES:
# CIK identifies a COMPANY, not a SECURITY. Alphabet's GOOG
# and GOOGL (both S&P 500 members) share one CIK, as do
# BRK-A / BRK-B. Using the bare CIK as a row key would
# collide those listings. We therefore build a security-level
# key:  sec_id = "<CIK>:<canonical ticker>"
# - The CIK part gives rename-stability across time
#   (ANTM and ELV resolve to the same CIK).
# - The canonical-ticker part keeps distinct share classes
#   distinct (GOOG vs GOOGL).
#
# TICKER_RENAME_MAP:
# The SEC file only maps CURRENT tickers. Old symbols of
# renamed companies are not in it, so we first canonicalise
# old→current with a manual map (extend as needed — this
# covers the well-known S&P 500 renames since 2020, and
# directly implements "Next Step 7" from the v16 notes).
#
# FALLBACK: tickers with no CIK match (delisted/acquired names
# Yahoo still returns, e.g. some SPACs) get a synthetic
# "NOCIK:<ticker>" id so the pipeline still works for them —
# they just don't gain rename-stability.
# =============================================================

print("Cell 2b: Building ticker → CIK entity mapping...")

# Old symbol → current symbol for known renames (NOT acquisitions).
# Extend this map as you discover more renamed tickers.
TICKER_RENAME_MAP = {
    "FB":    "META",   # Facebook → Meta (2022)
    "ANTM":  "ELV",    # Anthem → Elevance (2022)
    "DISCA": "WBD",    # Discovery A → Warner Bros. Discovery (2022)
    # NOTE: DISCK (Discovery class C) is deliberately NOT mapped —
    # DISCA and DISCK traded simultaneously; mapping both onto WBD
    # would collide two live securities onto one key. Map only the
    # primary class per company.
    "KORS":  "CPRI",   # Michael Kors → Capri
    "INFO":  "SPGI",   # IHS Markit merged into S&P Global (2022)
    "FISV":  "FI",     # Fiserv rename (2023)
    "PKI":   "RVTY",   # PerkinElmer → Revvity (2023)
    "RE":    "EG",     # Everest Re → Everest Group (2023)
    # v18 additions — 1:1 ticker renames of the SAME entity, added because
    # these old symbols failed to download in the v17 run. Each recovers
    # the entity's history under its current symbol via the Cell 3
    # fallback. VERIFY these before publishing (added from memory of
    # corporate actions; acquisitions/mergers with a different surviving
    # entity are deliberately NOT mapped):
    "BLL":   "BALL",   # Ball Corp rename (2022)
    "WLTW":  "WTW",    # Willis Towers Watson rename (2022)
    "CTL":   "LUMN",   # CenturyLink → Lumen (2020)
    "FBHS":  "FBIN",   # Fortune Brands rename (2022)
    "FLT":   "CPAY",   # FLEETCOR → Corpay (2024)
    "PEAK":  "DOC",    # Healthpeak ticker change (2024)
    "NLOK":  "GEN",    # NortonLifeLock → Gen Digital (2022)
    "CDAY":  "DAY",    # Ceridian → Dayforce (2024)
    "COG":   "CTRA",   # Cabot → Coterra (2021)
    "ADS":   "BFH",    # Alliance Data → Bread Financial (2022)
    "GPS":   "GAP",    # Gap ticker change (2024)
}

def canonical_ticker(t):
    """Map an (old) ticker to its current canonical symbol."""
    t = str(t).strip().upper().replace(".", "-")
    return TICKER_RENAME_MAP.get(t, t)

# --- Fetch the SEC's official ticker → CIK file ---
# SEC requires a descriptive User-Agent header on all requests.
TICKER_TO_CIK = {}
try:
    resp = requests.get(
        "https://www.sec.gov/files/company_tickers.json",
        headers={"User-Agent": "alpha-research-project contact@example.com"},
        timeout=30,
    )
    resp.raise_for_status()
    sec_map = resp.json()
    for _, rec in sec_map.items():
        tk = str(rec["ticker"]).strip().upper()
        cik10 = f"{int(rec['cik_str']):010d}"
        # store both dash and dot variants of the symbol
        TICKER_TO_CIK[tk] = cik10
        TICKER_TO_CIK[tk.replace(".", "-")] = cik10
        TICKER_TO_CIK[tk.replace("-", ".")] = cik10
    print(f"Cell 2b: SEC mapping loaded — {len(sec_map):,} companies.")
except Exception as e:
    print(f"Cell 2b: WARNING — SEC ticker/CIK fetch failed ({e}).")
    print("Cell 2b: Falling back to synthetic ids (pipeline still runs,")
    print("Cell 2b: but without cross-rename entity stability).")

def get_cik(ticker):
    """Pure 10-digit CIK for a ticker (after canonicalisation), or None."""
    return TICKER_TO_CIK.get(canonical_ticker(ticker))

def get_sec_id(ticker):
    """
    Stable SECURITY-level entity id used as the join/group key
    everywhere downstream (replaces raw ticker as the entity key):
        "<CIK>:<canonical_ticker>"  when CIK is known
        "NOCIK:<canonical_ticker>"  otherwise
    """
    ct  = canonical_ticker(ticker)
    cik = TICKER_TO_CIK.get(ct)
    return f"{cik}:{ct}" if cik else f"NOCIK:{ct}"

# Quick coverage report on the download universe
_n_mapped = sum(get_cik(t) is not None for t in all_tickers_ever)
print(f"Cell 2b: CIK coverage on universe: {_n_mapped}/{len(all_tickers_ever)} tickers"
      f" ({_n_mapped/len(all_tickers_ever):.1%})")
print("Cell 2b: Unmapped tickers keep a synthetic 'NOCIK:<ticker>' id.")
print("Cell 2b: Entity key ready — downstream cells group/join on 'sec_id'.")

In [ ]:
# =============================================================
# Cell 3 - Price Data Download  (cached)
# =============================================================
# The reshaped raw long price panel is cached to Data/md_raw_long.parquet.
# First run downloads from Yahoo and writes the cache; later runs load it
# (no network, no rate limits). Delete the file to force a fresh pull.
# =============================================================

PATH_RAW_PRICES = "Data/md_raw_long.parquet"

if os.path.exists(PATH_RAW_PRICES):
    price_data = pd.read_parquet(PATH_RAW_PRICES)
    price_data["date"] = pd.to_datetime(price_data["date"]).astype("datetime64[ns]")
    price_data = price_data.sort_values(["ticker", "date"]).reset_index(drop=True)
    print(f"Cell 3: Loaded cached raw price panel {price_data.shape} from {PATH_RAW_PRICES}")
    print(f"Cell 3: Tickers loaded: {price_data['ticker'].nunique()}")
else:
    # =============================================================
    # Cell 3 — Price Data Download
    # =============================================================
    # We download adjusted OHLCV data for ALL tickers that have
    # EVER been in the S&P 500 since START_DATE (not just current
    # members). This is required for the point-in-time universe
    # to work correctly — we need returns data for stocks that
    # were later removed from the index.
    #
    # This will download more tickers than v10 (~550-600 vs ~503).
    #
    # WHY AUTO-ADJUST=TRUE:
    # Adjusted prices account for dividends and splits. Without
    # this, a 2:1 stock split would appear as a -50% return,
    # contaminating momentum and return features.
    # =============================================================

    print("Cell 3: Downloading price data for all historical S&P 500 tickers...")
    print(f"Cell 3: Tickers to download: {len(all_tickers_ever)}")
    print("Cell 3: This may take 3-5 minutes...")

    # v18: also download each ticker's CANONICAL (current) symbol so that
    # renamed names whose old symbol Yahoo no longer serves (e.g. BLL →
    # BALL) are recovered from the current symbol's history.
    download_tickers = sorted(
        set(all_tickers_ever) | {canonical_ticker(t) for t in all_tickers_ever}
    )
    print(f"Cell 3: Including canonical symbols → {len(download_tickers)} downloads")

    raw = yf.download(
        download_tickers,
        start=START_DATE,
        auto_adjust=True,
        group_by="ticker",
        threads=True,
        progress=False
    )

    print("Cell 3: Download complete. Reshaping to long panel format...")

    price_list = []
    skipped = []

    for t in tqdm(all_tickers_ever, desc="Cell 3: Reshaping"):
        try:
            # v18: try the original symbol first, then its canonical rename.
            # Rows keep ticker = t (the ORIGINAL symbol) so the point-in-time
            # membership filter in Cell 4 still matches the history file;
            # entity identity is unified later via sec_id.
            df = None
            for src in (t, canonical_ticker(t)):
                if src not in raw.columns.get_level_values(0):
                    continue
                cand = raw[src].copy().reset_index()
                cand.rename(columns={"Date": "date"}, inplace=True)
                cand.columns = [c.lower() for c in cand.columns]
                if "close" in cand.columns and not cand["close"].isna().all():
                    df = cand
                    break
            if df is None:
                skipped.append(t)
                continue
            df["ticker"] = t
            price_list.append(df[["date", "ticker", "open", "high", "low", "close", "volume"]])
        except Exception as e:
            skipped.append(t)

    # ── v19: one retry pass for skipped tickers ──
    # The v18 run lost live large-caps (BK, MMC, K, CTRA, DAY, FI, ...) to
    # transient Yahoo failures — that is avoidable survivorship. Retry each
    # skipped ticker (and its canonical symbol) once, individually.
    if skipped:
        print(f"Cell 3: Retrying {len(skipped)} skipped tickers individually...")
        still_skipped = []
        for t in tqdm(skipped, desc="Cell 3: Retry"):
            got = False
            for src in dict.fromkeys([t, canonical_ticker(t)]):
                try:
                    cand = yf.download(src, start=START_DATE, auto_adjust=True,
                                       progress=False)
                    if cand is None or cand.empty:
                        continue
                    cand = cand.reset_index()
                    if isinstance(cand.columns, pd.MultiIndex):
                        cand.columns = [c[0] for c in cand.columns]
                    cand.rename(columns={"Date": "date"}, inplace=True)
                    cand.columns = [str(c).lower() for c in cand.columns]
                    if "close" in cand.columns and not cand["close"].isna().all():
                        cand["ticker"] = t
                        price_list.append(
                            cand[["date", "ticker", "open", "high", "low",
                                  "close", "volume"]])
                        got = True
                        break
                except Exception:
                    continue
            if not got:
                still_skipped.append(t)
        print(f"Cell 3: Retry recovered {len(skipped) - len(still_skipped)} tickers;"
              f" {len(still_skipped)} remain missing (mostly acquired/delisted:")
        print(f"Cell 3: {sorted(still_skipped)})")
        skipped = still_skipped

    price_data = pd.concat(price_list, ignore_index=True)
    price_data["date"] = pd.to_datetime(price_data["date"]).astype("datetime64[ns]")
    price_data = price_data.sort_values(["ticker", "date"]).reset_index(drop=True)

    print(f"Cell 3: Panel shape: {price_data.shape}")
    print(f"Cell 3: Tickers loaded: {price_data['ticker'].nunique()}")
    print(f"Cell 3: Tickers skipped (no data): {len(skipped)}")
    print(f"Cell 3: Date range: {price_data['date'].min().date()} → {price_data['date'].max().date()}")

    price_data.to_parquet(PATH_RAW_PRICES, index=False)
    print(f"Cell 3: Cached raw price panel -> {PATH_RAW_PRICES}")


In [ ]:
# =============================================================
# Cell 4 — Apply Point-in-Time Universe Filter
# =============================================================
# Now that we have price data for all historical tickers,
# we apply the point-in-time filter: for each row in the
# price panel, we check whether that ticker was actually in
# the S&P 500 on that date.
#
# If it wasn't (either not yet added, or already removed),
# the row is EXCLUDED from the panel.
#
# This is the core survivorship bias fix.
# Stocks removed from the index stay in the dataset only
# for the period they were actually members — including
# their bad performance around removal time.
#
# RESULT: price_panel_pit (point-in-time filtered panel)
# This replaces the naive price_panel from v10.
# =============================================================

print("Cell 4: Applying point-in-time universe filter...")

def is_in_universe(row):
    """Check if ticker was in the S&P 500 on the given date."""
    date = row["date"]
    ticker = row["ticker"]
    # find most recent constituent set on or before this date
    ts = pit_universe.get(date)
    if ts is None:
        # find nearest prior date
        prior_dates = [d for d in pit_universe.keys() if d <= date]
        if not prior_dates:
            return False
        ts = pit_universe[max(prior_dates)]
    return ticker in ts

# Vectorised version — much faster than row-apply
# Build a date→ticker validity DataFrame
print("Cell 4: Building daily valid ticker sets (vectorised)...")

# For each price row, check membership
# Strategy: merge price_data dates with daily_constituents lookup
price_data["date_norm"] = pd.to_datetime(price_data["date"]).dt.normalize()

# Create a long-format valid ticker list for merging
valid_rows = []
for _, row in tqdm(daily_constituents.iterrows(),
                   total=len(daily_constituents),
                   desc="Cell 4: Expanding constituent sets"):
    for t in row["ticker_set"]:
        valid_rows.append({"date": row["date"], "ticker": t})

valid_df = pd.DataFrame(valid_rows)
valid_df["date"] = pd.to_datetime(valid_df["date"])

# Forward-fill: each business day inherits the previous change date's list
# Use merge_asof per ticker would be slow; instead build a daily boolean mask
# by merging price data with the last known constituent set per date

# Get the most recent constituent set date for each price data date
price_dates = price_data[["date_norm"]].drop_duplicates().rename(columns={"date_norm": "date"})
price_dates = price_dates.sort_values("date")

# merge_asof: for each price date, find the most recent constituent change
date_to_change = pd.merge_asof(
    price_dates,
    daily_constituents[["date"]].rename(columns={"date": "change_date"}),
    left_on="date",
    right_on="change_date",
    direction="backward"
)
date_to_change = date_to_change.set_index("date")["change_date"].to_dict()

# Now filter: keep row if ticker in constituent set of its change_date
print("Cell 4: Filtering price rows by point-in-time membership...")

def get_valid_set(date):
    change_date = date_to_change.get(date)
    if change_date is None:
        return set()
    return pit_universe.get(change_date, set())

# Build mask — group by date for efficiency
# v17 FIX: the mask is built in DATE-major order (groupby iterates by
# date), but price_data rows are in TICKER-major order (Cell 3 sort).
# Assigning a plain list is positional and would misalign the mask with
# the rows. We therefore collect the row indices alongside the mask and
# assign an index-aligned Series instead.
keep_mask = []
row_index = []
for date, group in tqdm(price_data.groupby("date_norm"),
                        desc="Cell 4: Filtering"):
    valid_set = get_valid_set(date)
    keep_mask.extend(t in valid_set for t in group["ticker"])
    row_index.extend(group.index)

price_data["in_universe"] = pd.Series(keep_mask, index=row_index)
price_panel_pit = price_data[price_data["in_universe"]].drop(
    columns=["date_norm", "in_universe"]
).reset_index(drop=True)

print(f"Cell 4: Original rows: {len(price_data):,}")
print(f"Cell 4: After PIT filter: {len(price_panel_pit):,}")
print(f"Cell 4: Rows removed (not in index on that date): {len(price_data) - len(price_panel_pit):,}")
print(f"Cell 4: Unique tickers in filtered panel: {price_panel_pit['ticker'].nunique()}")
print("Cell 4: Point-in-time price panel ready.")

# ── v17 (change #1): attach the stable CIK-based entity key ──
# From here on, every per-name groupby / join in the pipeline uses
# sec_id (Cell 2b) instead of the raw ticker, so renamed companies
# keep one continuous history.
price_panel_pit["cik"]    = price_panel_pit["ticker"].map(get_cik)
price_panel_pit["sec_id"] = price_panel_pit["ticker"].map(get_sec_id)

# Safety: if an old and a new symbol of the same entity ever appear on
# the same date (can happen right around a rename), keep one row per
# (date, sec_id) so downstream joins stay unique.
_n_before = len(price_panel_pit)
price_panel_pit = (
    price_panel_pit.sort_values(["sec_id", "date"])
    .drop_duplicates(subset=["date", "sec_id"], keep="last")
    .reset_index(drop=True)
)
print(f"Cell 4: Entity key attached — {price_panel_pit['sec_id'].nunique()} sec_ids"
      f" for {price_panel_pit['ticker'].nunique()} tickers.")
print(f"Cell 4: Duplicate (date, sec_id) rows removed: {_n_before - len(price_panel_pit)}")
_n_nocik = price_panel_pit.loc[price_panel_pit["cik"].isna(), "ticker"].nunique()
print(f"Cell 4: Tickers without a CIK match (synthetic NOCIK id): {_n_nocik}")

In [ ]:
# =============================================================
# Cell 5 — Equity Feature Engineering
# =============================================================
# Build standard cross-sectional equity signals from the
# point-in-time filtered price panel.
#
# FEATURES:
# - ret:        1-day return
# - mom_5/20/60/252: momentum signals (trend following)
# - vol_20/60:  realised close-to-close volatility (risk signal)
# - ma_spread:  price vs 20-day MA (mean-reversion signal)
# - yz_vol_20:  Yang-Zhang range-based OHLC volatility (NEW v17)
# - adv_20:     20-day average daily DOLLAR volume (NEW v17)
#               → used for the σ/√ADV liquidity term in the
#                 optimizer and cost model, NOT as a model feature
#
# YANG-ZHANG VOLATILITY (NEW in v17, requested change #4):
# Close-to-close vol throws away intraday information. The
# Yang-Zhang estimator combines three components:
#   σ²_YZ = σ²_overnight + k·σ²_open-to-close + (1−k)·σ²_RS
# where σ²_RS is the Rogers-Satchell range estimator and
#   k = 0.34 / (1.34 + (n+1)/(n−1)).
# It is drift-independent, handles overnight gaps, and is the
# minimum-variance unbiased estimator in this family — several
# times more efficient than close-to-close vol at the same
# window length. We use a 20-day window so it is directly
# comparable to vol_20 (both are DAILY vols, not annualised).
#
# ENTITY KEY (NEW in v17, requested change #1):
# All per-name calculations now group by sec_id (CIK-based,
# Cell 2b) instead of raw ticker, so a renamed company's
# history (ANTM→ELV, FB→META, ...) is treated as ONE
# continuous series instead of breaking at the rename date.
# CAVEAT: across a rename boundary the two Yahoo price series
# may not be perfectly back-adjusted onto each other, so
# long-horizon features (mom_252) spanning the boundary can
# carry a level artifact for those few names — still better
# than discarding the pre-rename history entirely.
#
# All computed per-entity via groupby — no cross-name leakage.
# Lagging is applied later in Cell 11 (strict separation).
# =============================================================

print("Cell 5: Building equity features from point-in-time price panel...")

def build_equity_features(df):
    """
    Build momentum, volatility (close-to-close AND Yang-Zhang),
    MA-spread, and dollar-ADV columns.
    Uses groupby("sec_id") so every calculation is per-entity
    (CIK-stable across ticker renames).
    """
    df = df.sort_values(["sec_id", "date"]).copy()
    g  = df.groupby("sec_id")["close"]

    df["ret"]     = g.pct_change(1)
    df["mom_5"]   = g.pct_change(5)
    df["mom_20"]  = g.pct_change(20)
    df["mom_60"]  = g.pct_change(60)
    df["mom_252"] = g.pct_change(252)

    ret_g        = df.groupby("sec_id")["ret"]
    df["vol_20"] = ret_g.rolling(20, min_periods=15).std().reset_index(level=0, drop=True)
    df["vol_60"] = ret_g.rolling(60, min_periods=45).std().reset_index(level=0, drop=True)

    ma20         = g.rolling(20, min_periods=15).mean().reset_index(level=0, drop=True)
    df["ma_spread"] = df["close"] / ma20 - 1

    # ── Yang-Zhang range-based volatility (20-day, daily units) ──
    prev_close = g.shift(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        df["_yz_on"] = np.log(df["open"]  / prev_close)          # overnight
        df["_yz_oc"] = np.log(df["close"] / df["open"])          # open→close
        df["_yz_rs"] = (np.log(df["high"] / df["open"])          # Rogers-Satchell
                        * np.log(df["high"] / df["close"])
                        + np.log(df["low"] / df["open"])
                        * np.log(df["low"] / df["close"]))
    df[["_yz_on", "_yz_oc", "_yz_rs"]] = (
        df[["_yz_on", "_yz_oc", "_yz_rs"]].replace([np.inf, -np.inf], np.nan)
    )

    n_yz  = 20
    k_yz  = 0.34 / (1.34 + (n_yz + 1) / (n_yz - 1))
    gsec  = df.groupby("sec_id")
    var_on = gsec["_yz_on"].rolling(n_yz, min_periods=15).var().reset_index(level=0, drop=True)
    var_oc = gsec["_yz_oc"].rolling(n_yz, min_periods=15).var().reset_index(level=0, drop=True)
    rs_mu  = gsec["_yz_rs"].rolling(n_yz, min_periods=15).mean().reset_index(level=0, drop=True)
    df["yz_vol_20"] = np.sqrt(
        (var_on + k_yz * var_oc + (1 - k_yz) * rs_mu).clip(lower=0)
    )
    df = df.drop(columns=["_yz_on", "_yz_oc", "_yz_rs"])

    # ── Average daily dollar volume (liquidity, for σ/√ADV terms) ──
    df["dollar_vol"] = df["close"] * df["volume"]
    df["adv_20"] = (
        df.groupby("sec_id")["dollar_vol"]
        .rolling(20, min_periods=15).mean()
        .reset_index(level=0, drop=True)
    )

    return df

price_panel = build_equity_features(price_panel_pit)

print(f"Cell 5: Feature panel shape: {price_panel.shape}")
print(f"Cell 5: Entities (sec_id): {price_panel['sec_id'].nunique()}"
      f"  |  Tickers: {price_panel['ticker'].nunique()}")
print("Cell 5: Null counts per feature (raw, before lagging):")
feat_cols = ["ret","mom_5","mom_20","mom_60","mom_252",
             "vol_20","vol_60","ma_spread","yz_vol_20","adv_20"]
print(price_panel[feat_cols].isna().sum().to_string())
print("\nCell 5: vol_20 vs yz_vol_20 sanity check (should be same order of magnitude):")
_chk = price_panel[["vol_20","yz_vol_20"]].dropna()
if len(_chk) > 0:
    print(f"  median vol_20    = {_chk['vol_20'].median():.5f}")
    print(f"  median yz_vol_20 = {_chk['yz_vol_20'].median():.5f}")
    print(f"  correlation      = {_chk['vol_20'].corr(_chk['yz_vol_20']):+.3f}")

In [ ]:
# =============================================================
# Cell 5b — Hurst Exponent, rolling 6 months (NEW in v17)
# =============================================================
# WHAT IT MEASURES:
# The Hurst exponent H of a price series characterises its
# memory structure over a window:
#   H > 0.5 → trending / persistent   (momentum-friendly)
#   H = 0.5 → random walk             (no exploitable memory)
#   H < 0.5 → mean-reverting          (reversal-friendly)
# As a cross-sectional feature it separates stocks currently in
# a trending regime from those in a choppy/mean-reverting one —
# plausibly useful interacting with the momentum features.
#
# METHOD (aggregated-variance estimator):
# For lags q ∈ {2,4,8,16,32}, compute the std of q-day log-price
# differences; H is the slope of log(std) on log(q). This is the
# standard fast estimator (R/S analysis gives similar values but
# is slower and noisier at this window length).
#
# WINDOW: 126 trading days ≈ 6 months, per the request.
#
# RUNTIME CONTROL:
# A full rolling-apply (≈570 entities × ~1,300 days) is slow, so
# H is computed every HURST_STEP=5 days per entity and forward-
# filled in between. H drifts slowly, so a 1-week refresh loses
# essentially nothing and cuts runtime ~5x.
# =============================================================

print("Cell 5b: Computing rolling 6-month Hurst exponent per entity...")

HURST_WINDOW = 126   # ~6 months of trading days
HURST_STEP   = 5     # recompute weekly, ffill in between
_HURST_LAGS  = np.array([2, 4, 8, 16, 32])
_LOG_LAGS    = np.log(_HURST_LAGS)

def _hurst_from_logprice(logp):
    """Aggregated-variance Hurst estimator on a log-price window."""
    taus = np.empty(len(_HURST_LAGS))
    for j, q in enumerate(_HURST_LAGS):
        d = logp[q:] - logp[:-q]
        taus[j] = d.std()
    if np.any(~np.isfinite(taus)) or np.any(taus <= 0):
        return np.nan
    # slope of log(std) vs log(lag)
    return np.polyfit(_LOG_LAGS, np.log(taus), 1)[0]

price_panel = price_panel.sort_values(["sec_id", "date"])
price_panel["hurst_126"] = np.nan

_n_done = 0
for sec, g in tqdm(price_panel.groupby("sec_id"), desc="Cell 5b: Hurst"):
    close = g["close"].values.astype(float)
    if len(close) < HURST_WINDOW + 1 or np.any(close <= 0):
        continue
    logp = np.log(close)
    out  = np.full(len(logp), np.nan)
    for end in range(HURST_WINDOW, len(logp) + 1, HURST_STEP):
        out[end - 1] = _hurst_from_logprice(logp[end - HURST_WINDOW:end])
    # forward-fill between weekly recomputations (within entity only)
    s = pd.Series(out, index=g.index).ffill()
    price_panel.loc[g.index, "hurst_126"] = s
    _n_done += 1

_valid = price_panel["hurst_126"].dropna()
print(f"Cell 5b: Hurst computed for {_n_done} entities.")
print(f"Cell 5b: Non-null hurst_126 rows: {len(_valid):,}")
if len(_valid) > 0:
    print(f"Cell 5b: Distribution — mean={_valid.mean():.3f}  "
          f"p25={_valid.quantile(0.25):.3f}  p75={_valid.quantile(0.75):.3f}")
    print("Cell 5b: (≈0.5 = random walk; >0.5 trending; <0.5 mean-reverting)")
print("Cell 5b: hurst_126 will be lagged in Cell 11 and modelled as a factor.")

In [ ]:
# =============================================================
# Cell 6 — Macro Data (FRED)
# =============================================================
# Four FRED macro series providing economic regime context.
#
# SERIES:
# - CPIAUCSL (cpi):       Consumer Price Index (inflation)
# - UNRATE (unrate):      Unemployment rate (economic cycle)
# - DGS10 (dgs10):        10-year Treasury yield (long rates)
# - DGS2 (dgs2):          2-year Treasury yield (short rates)
# - yield_spread:         DGS10 - DGS2 (yield curve slope)
#
# POINT-IN-TIME NOTE:
# We lag by 1 day in Cell 9. FRED data has real-world
# publication lags (e.g. CPI ~2 weeks after month end).
# A production system would use vintage release dates.
# Forward-filling propagates last known value to all days.
# =============================================================

print("Cell 6: Fetching macro series from FRED...")

fred = Fred(api_key=FRED_API_KEY)

def get_fred_series(series_id, name):
    """Fetch a FRED series and return as a named DataFrame."""
    try:
        s  = fred.get_series(series_id, observation_start=START_DATE)
        df = s.to_frame(name=name)
        df.index.name = "date"
        df.index = pd.to_datetime(df.index).astype("datetime64[ns]")
        print(f"  Cell 6: ✓ {series_id} ({name}): {len(df)} observations")
        return df
    except Exception as e:
        print(f"  Cell 6: ✗ {series_id} failed: {e}")
        return None

series_to_fetch = {
    "CPIAUCSL": "cpi",
    "UNRATE":   "unrate",
    "DGS10":    "dgs10",
    "DGS2":     "dgs2",
}

macro_parts = []
for sid, name in series_to_fetch.items():
    df = get_fred_series(sid, name)
    if df is not None:
        macro_parts.append(df)

macro_raw = macro_parts[0].join(macro_parts[1:], how="outer")
macro_raw = macro_raw.sort_index().ffill()
macro_raw["yield_spread"] = macro_raw["dgs10"] - macro_raw["dgs2"]

print(f"Cell 6: Macro dataset shape: {macro_raw.shape}")
print(f"Cell 6: Date range: {macro_raw.index.min().date()} → {macro_raw.index.max().date()}")
print(f"Cell 6: Null counts after ffill: {macro_raw.isna().sum().sum()} (should be 0)")

In [ ]:
# =============================================================
# Cell 7 — ETF Regime Features
# =============================================================
# Market regime signals from ETF daily returns.
#
# FEATURES:
# - spy_tlt:  SPY - TLT return (risk-on vs risk-off)
# - hyg_tlt:  HYG - TLT return (credit appetite)
# - gold_ret: GLD daily return (inflation / stress signal)
# - oil_ret:  USO daily return (commodity / growth signal)
#
# WHY RETURNS NOT LEVELS:
# ETF levels are non-stationary (trending), violating OLS
# assumptions. Daily returns are stationary and directly
# interpretable as regime shocks.
#
# IMPORTANT: these are market-wide signals (same value for
# all stocks on a given day). They are NOT cross-sectionally
# ranked — ranking would make every stock get ~0.5.
# =============================================================

print("Cell 7: Downloading ETF regime features...")

etf_tickers = ["SPY", "TLT", "HYG", "GLD", "USO"]
etf_raw  = yf.download(etf_tickers, start=START_DATE, auto_adjust=True, progress=False)["Close"]
etf_ret  = etf_raw.pct_change()

etf_features = pd.DataFrame(index=etf_ret.index)
etf_features.index = pd.to_datetime(etf_features.index).astype("datetime64[ns]")
etf_features.index.name = "date"
etf_features["spy_tlt"]  = etf_ret["SPY"] - etf_ret["TLT"]
etf_features["hyg_tlt"]  = etf_ret["HYG"] - etf_ret["TLT"]
etf_features["gold_ret"] = etf_ret["GLD"]
etf_features["oil_ret"]  = etf_ret["USO"]
etf_features = etf_features.sort_index().ffill()

print(f"Cell 7: ETF feature shape: {etf_features.shape}")
print(f"Cell 7: Date range: {etf_features.index.min().date()} → {etf_features.index.max().date()}")
print(f"Cell 7: Null counts: {etf_features.isna().sum().sum()} (should be 0)")

In [ ]:
# =============================================================
# Cell 8 — Short Volume Features
# =============================================================
# FINRA daily short volume data — a KEY upgrade from v10.
#
# SHORT VOLUME vs SHORT INTEREST (important distinction):
# - Short VOLUME (this cell): daily FLOW — how much was
#   shorted today relative to total volume. High short volume
#   ratio means heavy bearish activity TODAY.
# - Short INTEREST (Cell 9): total STOCK — how many shares
#   are currently held short (not yet covered). Changes slowly.
#
# FEATURES USED:
# - short_ratio:      short_volume / total_volume (raw signal)
#   Mean ~0.49 (49% of FINRA volume is short on avg — normal)
# - short_ratio_z20: 20-day z-score of short_ratio
#   (pre-computed: how unusual is today's short activity?)
#   POSITIVE z-score = unusually high short selling = bearish signal
# - short_ratio_chg5: 5-day change in short_ratio
#   (momentum of short selling activity)
#
# COVERAGE: 2021-01-04 onwards, 728 symbols.
# Dates before this will have NaN for these features
# (handled by dropna in modeling cells).
#
# POINT-IN-TIME: short volume is published daily by FINRA
# with a 1-day lag. We apply an additional shift(1) in Cell 10.
# =============================================================

print("Cell 8: Loading short volume data...")

sv = pd.read_parquet(PATH_SHORT_VOL)
sv["date"]   = pd.to_datetime(sv["date"]).astype("datetime64[ns]")
sv["symbol"] = sv["symbol"].str.strip()

# We use three features from this file
sv_features = sv[["date", "symbol", "short_ratio",
                   "short_ratio_z20", "short_ratio_chg5"]].copy()

# Rename symbol → ticker for consistent joining
sv_features = sv_features.rename(columns={"symbol": "ticker"})

# v17 (change #1): CIK-based entity key so the join in Cell 11 survives
# ticker renames (old symbols in this file map to the same sec_id as
# the current symbol in the price panel).
sv_features["sec_id"] = sv_features["ticker"].map(get_sec_id)
sv_features = sv_features.drop_duplicates(subset=["date", "sec_id"], keep="last")

print(f"Cell 8: Short volume rows: {len(sv_features):,}")
print(f"Cell 8: Date range: {sv_features['date'].min().date()} → {sv_features['date'].max().date()}")
print(f"Cell 8: Unique tickers: {sv_features['ticker'].nunique()}")
print("Cell 8: Null counts:")
print(sv_features[["short_ratio","short_ratio_z20","short_ratio_chg5"]].isna().sum().to_string())
print("Cell 8: short_ratio distribution:")
print(sv_features["short_ratio"].describe().to_string())

In [ ]:
# =============================================================
# Cell 9 — Short Interest Features
# =============================================================
# Bi-monthly (twice per month) short interest data.
#
# FEATURES (computed for reference, NOT included in ALL_FEATURES):
# - days_to_cover: short_interest / avg_daily_volume
#   How many days would it take all short sellers to cover?
#   HIGH value = high squeeze risk / overcrowded short
#
# - short_interest_rank: cross-sectional rank of short interest
#
# IMPORTANT DECISION — EXCLUDED FROM ALL_FEATURES (Cell 12):
# Short interest data is NULL before December 2022 (only 180
# of 570+ tickers covered, starting partway through our backtest
# window). If included in ALL_FEATURES, the global dropna step
# in Cell 13 would drop EVERY row before Dec 2022 — destroying
# 3 years of otherwise-valid training data for momentum, vol,
# short volume, and fundamental features.
#
# We compute these columns and merge them into final_df for
# transparency and potential future use, but EXCLUDE them from
# the canonical ALL_FEATURES list used for modelling. This is
# the assignment's chosen tradeoff: prioritise dataset coverage
# (2020-2026) over including a thinly-covered signal.
#
# FORWARD FILL:
# Short interest is reported twice monthly. We forward-fill
# within each ticker so every trading day has the most
# recently reported value (as a live system would).
# =============================================================

print("Cell 9: Loading short interest data...")

si = pd.read_parquet(PATH_SHORT_INT)
si["date"]   = pd.to_datetime(si["date"]).astype("datetime64[ns]")
si["symbol"] = si["symbol"].str.strip()

si_features = si[["date", "symbol", "days_to_cover",
                   "short_interest_rank"]].copy()
si_features = si_features.rename(columns={"symbol": "ticker"})

# v17 (change #1): CIK-based entity key (see Cell 2b / Cell 8 note)
si_features["sec_id"] = si_features["ticker"].map(get_sec_id)
si_features = si_features.drop_duplicates(subset=["date", "sec_id"], keep="last")

# Forward fill within each ticker (bi-monthly → daily)
si_features = si_features.sort_values(["sec_id", "date"])
si_features[["days_to_cover", "short_interest_rank"]] = (
    si_features.groupby("sec_id")[["days_to_cover", "short_interest_rank"]].ffill()
)

non_null = si_features.dropna(subset=["days_to_cover"])
print(f"Cell 9: Short interest rows: {len(si_features):,}")
print(f"Cell 9: Date range: {si_features['date'].min().date()} → {si_features['date'].max().date()}")
print(f"Cell 9: Non-null rows: {len(non_null):,} (from Dec 2022 onwards)")
print(f"Cell 9: Unique tickers (non-null): {non_null['ticker'].nunique()}")
print("Cell 9: days_to_cover distribution (non-null):")
print(non_null["days_to_cover"].describe().to_string())
print("\nCell 9: DECISION — excluded from ALL_FEATURES (see Cell 12).")
print("Cell 9: Coverage too sparse (Dec 2022+, 180 tickers) to include")
print("Cell 9: without destroying 2020-2022 training data via dropna.")

In [ ]:
# =============================================================
# Cell 10 — Fundamental Features
# =============================================================
# Quarterly financial statement data, forward-filled to daily.
#
# RAW DATA COLUMNS: revenue, net_income, eps_diluted,
#                   assets, equity
#
# DERIVED FEATURES (all four chosen):
# 1. roe (Return on Equity) = net_income / equity
#    QUALITY signal: high ROE = efficient use of capital
#    Classic Fama-French profitability factor proxy
#
# 2. profit_margin = net_income / revenue
#    QUALITY signal: how much of each $ of revenue becomes profit
#    Stable margins = defensive moat, expanding = improving quality
#
# 3. revenue_growth = revenue / revenue.shift(4 quarters) - 1
#    GROWTH/MOMENTUM signal: year-over-year revenue growth
#    Strong growth firms tend to continue outperforming
#    (earnings momentum is well-documented in academic literature)
#
# 4. debt_to_equity = (assets - equity) / equity
#    VALUE/RISK signal: leverage ratio
#    High D/E = financial risk, tends to underperform in stress
#
# POINT-IN-TIME HANDLING:
# Fundamental data updates quarterly on earnings dates.
# It is already forward-filled to daily frequency in the source
# file (verified: AAPL has ~23 unique revenue values over 5 years,
# consistent with quarterly updates). This means the value on
# any given day is the MOST RECENTLY REPORTED figure — correct.
# We still apply a 1-day lag in Cell 11 as an extra safety measure.
#
# OUTLIER NOTE:
# Financial ratios can be extreme (negative equity → negative D/E,
# near-zero revenue → extreme margins). We winsorise in Cell 12.
# =============================================================

print("Cell 10: Loading and engineering fundamental features...")

fu = pd.read_parquet(PATH_FUNDAMENTALS)
fu["date"]   = pd.to_datetime(fu["date"]).astype("datetime64[ns]")
fu["symbol"] = fu["symbol"].str.strip()
fu = fu.rename(columns={"symbol": "ticker"})

# v17 (change #1): CIK-based entity key (see Cell 2b / Cell 8 note)
fu["sec_id"] = fu["ticker"].map(get_sec_id)
fu = fu.drop_duplicates(subset=["date", "sec_id"], keep="last")
fu = fu.sort_values(["sec_id", "date"]).reset_index(drop=True)

# ── Derived feature 1: ROE ──
# Cap: avoid extreme values when equity near zero
fu["roe"] = fu["net_income"] / fu["equity"].replace(0, np.nan)

# ── Derived feature 2: Profit Margin ──
fu["profit_margin"] = fu["net_income"] / fu["revenue"].replace(0, np.nan)

# ── Derived feature 3: Revenue Growth (YoY) ──
# shift(252) approximates 1 year of daily forward-filled data
# (4 quarters × ~63 trading days per quarter)
fu["revenue_growth"] = (
    fu.groupby("sec_id")["revenue"]
    .pct_change(252)   # compare to ~1 year ago
)

# ── Derived feature 4: Debt-to-Equity ──
fu["debt_to_equity"] = (fu["assets"] - fu["equity"]) / fu["equity"].replace(0, np.nan)

# Keep only the features we need
fu_features = fu[["date", "ticker", "sec_id", "roe", "profit_margin",
                   "revenue_growth", "debt_to_equity"]].copy()

print(f"Cell 10: Fundamentals rows: {len(fu_features):,}")
print(f"Cell 10: Date range: {fu_features['date'].min().date()} → {fu_features['date'].max().date()}")
print(f"Cell 10: Unique tickers: {fu_features['ticker'].nunique()}")
print("Cell 10: Null counts per feature:")
print(fu_features[["roe","profit_margin","revenue_growth","debt_to_equity"]].isna().sum().to_string())
print("Cell 10: ROE distribution (pre-winsorise):")
print(fu_features["roe"].describe().to_string())

In [ ]:
# =============================================================
# Cell 11 — Master Dataset Assembly + Strict Lag Discipline
# =============================================================
# THE GOLDEN RULE: only use information available at trade time.
#
# LAG STRATEGY:
# 1. Macro (Cell 6):       shift(1) before merge — yesterday's macro
# 2. ETF (Cell 7):         shift(1) before merge — yesterday's ETF
# 3. Short volume (Cell 8): shift(1) — FINRA data has 1-day pub lag
# 4. Short interest (Cell 9): shift(1) — bi-monthly, 1-day safety
# 5. Fundamentals (Cell 10): shift(1) — already forward-filled
#                             quarterly, 1-day extra safety
# 6. Stock features (mom/vol/ma): additional shift(1) per ticker
#    → we trade using YESTERDAY'S signal
# 7. Target: DEMEANED forward TARGET_HORIZON-day return (v18) — future-looking
#
# MERGE STRATEGY:
# - Macro/ETF: date-level left join (one value per date, all tickers)
# - Short vol/int/fundamentals: (date, ticker) left join
# - All macro/ETF/fundamental cols forward-filled within ticker
#   to handle sparse release schedules
# =============================================================

print("Cell 11: Building master dataset with strict lag discipline...")

# ── Step 1: Lag macro and ETF ──
print("  Cell 11: Lagging macro and ETF features by 1 day...")
macro_lagged = macro_raw.shift(1)
etf_lagged   = etf_features.shift(1)

# ── Step 2: Start from PIT-filtered price panel ──
final_df = price_panel.copy()
final_df["date"] = pd.to_datetime(final_df["date"]).astype("datetime64[ns]")

# ── Step 3: Merge macro ──
print("  Cell 11: Merging macro data...")
final_df = final_df.merge(
    macro_lagged.reset_index(),
    on="date", how="left"
)

# ── Step 4: Merge ETF ──
print("  Cell 11: Merging ETF regime features...")
final_df = final_df.merge(
    etf_lagged.reset_index(),
    on="date", how="left"
)

# ── Step 5: Merge short volume (date + ticker) ──
print("  Cell 11: Merging short volume features...")
sv_lagged = sv_features.copy()
sv_lagged = sv_lagged.sort_values(["sec_id","date"])
for col in ["short_ratio","short_ratio_z20","short_ratio_chg5"]:
    sv_lagged[col] = sv_lagged.groupby("sec_id")[col].shift(1)

# v17: join on (date, sec_id) — rename-stable — and drop the source
# file's ticker column to avoid _x/_y suffix collisions.
final_df = final_df.merge(
    sv_lagged.drop(columns=["ticker"]), on=["date","sec_id"], how="left"
)

# ── Step 6: Merge short interest (date + ticker) ──
print("  Cell 11: Merging short interest features...")
si_lagged = si_features.copy()
si_lagged = si_lagged.sort_values(["sec_id","date"])
for col in ["days_to_cover","short_interest_rank"]:
    si_lagged[col] = si_lagged.groupby("sec_id")[col].shift(1)

final_df = final_df.merge(
    si_lagged.drop(columns=["ticker"]), on=["date","sec_id"], how="left"
)

# ── Step 7: Merge fundamentals (date + ticker) ──
print("  Cell 11: Merging fundamental features...")
fu_lagged = fu_features.copy()
fu_lagged = fu_lagged.sort_values(["sec_id","date"])
for col in ["roe","profit_margin","revenue_growth","debt_to_equity"]:
    fu_lagged[col] = fu_lagged.groupby("sec_id")[col].shift(1)

final_df = final_df.merge(
    fu_lagged.drop(columns=["ticker"]), on=["date","sec_id"], how="left"
)

# ── Step 8: Forward fill sparse columns within ticker ──
print("  Cell 11: Forward-filling sparse columns within each ticker...")
sparse_fill_cols = [
    "cpi","unrate","dgs10","dgs2","yield_spread",
    "spy_tlt","hyg_tlt","gold_ret","oil_ret",
    "roe","profit_margin","revenue_growth","debt_to_equity",
    "days_to_cover","short_interest_rank"
]
final_df = final_df.sort_values(["sec_id","date"])
final_df[sparse_fill_cols] = (
    final_df.groupby("sec_id")[sparse_fill_cols].ffill()
)

# ── Step 9: Create _lag1 versions of stock-level features ──
print("  Cell 11: Creating _lag1 stock feature columns...")
stock_raw_cols = ["mom_5","mom_20","mom_60","mom_252",
                  "vol_20","vol_60","ma_spread",
                  "yz_vol_20","hurst_126"]   # v17: new technical factors
for col in stock_raw_cols:
    final_df[f"{col}_lag1"] = final_df.groupby("sec_id")[col].shift(1)

# Rename macro/ETF/short/fundamental cols to _lag1 for clarity
all_lag_rename = (
    ["cpi","unrate","dgs10","dgs2","yield_spread",
     "spy_tlt","hyg_tlt","gold_ret","oil_ret",
     "short_ratio","short_ratio_z20","short_ratio_chg5",
     "days_to_cover","short_interest_rank",
     "roe","profit_margin","revenue_growth","debt_to_equity"]
)
for col in all_lag_rename:
    if col in final_df.columns:
        final_df[f"{col}_lag1"] = final_df[col]

# ── Step 9b (v17, change #3): raw liquidity/vol copies for σ/√ADV ──
# tc_sigma = yesterday's 20-day close-to-close vol (daily units)
# tc_adv   = yesterday's 20-day average daily DOLLAR volume
# Lagged 1 day (known at trade time) but deliberately NOT winsorised
# or z-scored in Cell 13 — the optimizer (Cell 30) and the cost model
# (Cell 31) need them in raw units. They are NOT model features.
print("  Cell 11: Creating tc_sigma / tc_adv liquidity columns (raw units)...")
final_df["tc_sigma"] = final_df.groupby("sec_id")["vol_20"].shift(1)
final_df["tc_adv"]   = final_df.groupby("sec_id")["adv_20"].shift(1)

# ── Step 10 (v18): Construct target — DEMEANED forward return ──
# 10a. Raw forward returns per entity:
#      fwd_ret_1d = t → t+1 return  (used for PORTFOLIO PnL — the
#                   return a book formed at close t actually earns)
#      fwd_ret_H  = t → t+TARGET_HORIZON return (the PREDICTION horizon)
# 10b. target = fwd_ret_H minus its cross-sectional mean that date.
#      Demeaning removes the common market component from y, so no
#      model can score by predicting "the market goes up" — only by
#      ranking stocks against each other. (Per-date Spearman IC is
#      unchanged by a per-date constant shift, so IC numbers remain
#      directly comparable; what changes is what MSE-fitting rewards.)
print(f"  Cell 11: Constructing demeaned {TARGET_HORIZON}-day forward-return target...")
_g_close = final_df.groupby("sec_id")["close"]
final_df["fwd_ret_1d"] = _g_close.pct_change(1).shift(-1)
final_df["fwd_ret_h"]  = _g_close.pct_change(TARGET_HORIZON).shift(-TARGET_HORIZON)
final_df["target"] = (
    final_df["fwd_ret_h"]
    - final_df.groupby("date")["fwd_ret_h"].transform("mean")
)

final_df = final_df.sort_values(["date","ticker"]).reset_index(drop=True)

print(f"Cell 11: Master dataset shape: {final_df.shape}")
print(f"Cell 11: Tickers: {final_df['ticker'].nunique()}")
print(f"Cell 11: Date range: {final_df['date'].min().date()} → {final_df['date'].max().date()}")
print(f"Cell 11: Rows with valid target: {final_df['target'].notna().sum():,}")
print(f"Cell 11: Target = {TARGET_HORIZON}-day forward return, demeaned per date")
print(f"Cell 11: Cross-sectional mean of target (should be ~0): "
      f"{final_df['target'].mean():+.2e}")

In [ ]:
# =============================================================
# Cell 12 — Residualization Against Market Factor
# =============================================================
# WHY RESIDUALIZE:
# Stock-level features (momentum, vol, ma_spread) can partly
# just be re-expressing broad market movement rather than
# genuine stock-specific signal. For example, a stock's
# mom_60_lag1 might be high simply because the whole market
# rallied, not because that stock has idiosyncratic momentum.
#
# THE FIX (from course material on Residualization):
# Regress each stock feature against a market risk factor,
# keep only the RESIDUAL (the part NOT explained by the market).
#   X = stock feature (e.g. mom_60_lag1)
#   Z = market factor (spy_tlt_lag1, already in our dataset)
#   model = OLS(X ~ Z)
#   X_residualized = model.resid
#
# WHY THIS MATTERS FOR OUR RESULTS:
# Random Forest feature importance (Cell 19) tends to be
# dominated by spy_tlt_lag1 and hyg_tlt_lag1 — pure market and
# credit regime signals. If our stock features are correlated
# with those same signals, residualizing isolates genuine
# stock-specific alpha from "the whole market moved" noise.
#
# WHAT WE RESIDUALIZE:
# Only STOCK_FEATURES (momentum, vol, ma_spread) — these are
# the features plausibly contaminated by market beta.
# We do NOT residualize macro/ETF/short-vol/fundamental
# features — they are either already market-level signals
# themselves, or measure something structurally different
# (short selling activity, financial ratios) not expected to
# be redundant with market beta in the same way.
#
# METHOD: time-series regression, per ticker.
# For each ticker, regress that stock's feature against
# spy_tlt_lag1 ACROSS ALL DATES in its history, keep the
# residual. This preserves the interpretation: "does this
# stock's feature have a structural relationship to the
# market factor over time, and what's left over after
# removing that relationship (i.e. its market beta)?"
#
# IMPORTANT CORRECTION: an earlier version of this cell
# attempted a CROSS-SECTIONAL regression per date (grouping
# by date, regressing across stocks that day). That approach
# is mathematically invalid here: spy_tlt_lag1 is a market-wide
# value identical for every stock on a given day, so it has
# ZERO variance within any single date's cross-section. A
# regressor with no within-group variance cannot be regressed
# against, so every date was skipped and the residualized
# columns were entirely NaN. Grouping by ticker (so the
# regression uses variation in spy_tlt_lag1 ACROSS DAYS,
# which is real) fixes this.
#
# ORIGINAL VALUES PRESERVED:
# Pre-residualization values are kept in _raw columns in case
# they are needed for diagnostics or comparison later.
# =============================================================

print("Cell 12: Residualizing stock features against market factor (spy_tlt_lag1)...")

STOCK_FEATURES_RAW = [
    "mom_5_lag1","mom_20_lag1","mom_60_lag1","mom_252_lag1",
    "vol_20_lag1","vol_60_lag1","ma_spread_lag1",
    "yz_vol_20_lag1","hurst_126_lag1",   # v17: new technical factors
]

def residualize_against_market(df, feature_col, market_col="spy_tlt_lag1"):
    """
    Time-series residualization: for each ticker, regress
    feature_col on market_col across all dates in that
    ticker's history, return the residuals (same index as input).
    """
    resid = pd.Series(index=df.index, dtype=float)
    n_success = 0
    n_failed = 0

    for sec, idx in df.groupby("sec_id").groups.items():   # v17: entity key
        sub = df.loc[idx, [feature_col, market_col]].dropna()
        if len(sub) < 30:
            n_failed += 1
            continue  # not enough history for this ticker to regress reliably
        if sub[market_col].nunique() < 2:
            n_failed += 1
            continue  # market factor constant across this ticker's history (shouldn't happen, but safe)

        X = sm.add_constant(sub[market_col])
        y = sub[feature_col]
        try:
            model = sm.OLS(y, X).fit()
            resid.loc[sub.index] = model.resid
            n_success += 1
        except Exception:
            n_failed += 1
            continue

    print(f"    -> {feature_col}: {n_success} entities succeeded, {n_failed} entities failed")
    return resid

print("Cell 12: This should run quickly (per-entity OLS regression, ~570 sec_ids).")

residualized_count = 0
for col in STOCK_FEATURES_RAW:
    if col not in final_df.columns:
        continue
    print(f"  Cell 12: Residualizing {col}...")
    resid_col = residualize_against_market(final_df, col)
    # Keep original as backup, replace working column with residual
    final_df[f"{col}_raw"] = final_df[col]
    final_df[col] = resid_col
    residualized_count += 1

print(f"\nCell 12: Residualized {residualized_count} stock features against spy_tlt_lag1.")
print("Cell 12: Original (pre-residualization) values preserved in '_raw' columns.")

print("\nCell 12: Sanity check — correlation of residualized features with market factor")
print("Cell 12: (should be near 0, confirming market component was removed):")
for col in STOCK_FEATURES_RAW:
    if col in final_df.columns:
        valid = final_df[[col, "spy_tlt_lag1"]].dropna()
        if len(valid) > 100:
            corr = valid[col].corr(valid["spy_tlt_lag1"])
            print(f"  {col:<20} corr with market factor: {corr:+.5f}")

In [ ]:
# =============================================================
# Cell 12b — OPTIONAL: Residualize Against Fundamental Factors
#            (NEW in v17 — off by default)
# =============================================================
# IDEA (requested change #5):
# Instead of using fundamentals (ROE, margin, revenue growth,
# D/E) as direct model features, use them as RISK FACTORS:
# regress every stock-level technical feature on the fundamental
# factors and keep only the residual. The model then trades the
# part of momentum/vol/Hurst/etc. that is NOT explained by
# fundamental characteristics.
#
# CONSEQUENCE FOR THE FEATURE SET:
# Once features are residualized against fundamentals, the
# fundamental columns themselves MUST be dropped from
# ALL_FEATURES — otherwise the model sees both the factor and
# the factor-orthogonal features, which reintroduces exactly
# the collinearity the residualization removed, and the design
# stops being "a feature set that doesn't contain fundamental
# data". Cell 13 handles this automatically via the flag below:
# when the flag is True, ALL_FEATURES = stock (residualized)
# + macro/ETF + short-volume features only.
#
# METHOD — cross-sectional, per date (and why that's valid here):
# Unlike spy_tlt_lag1 (identical for every stock on a day, so
# zero within-day variance), fundamental features DIFFER across
# stocks on the same date, so a per-date cross-sectional OLS is
# well-posed:
#     for each date t:  feature_i = a + b·fund_i + e_i,
#     keep e_i (the fundamentally-neutral part of the feature).
# This is the standard "characteristic-neutralization" used for
# sector/size/value neutral alphas.
#
# ORDERING NOTE: this runs AFTER the market-factor
# residualization (Cell 12), i.e. features are first market-
# neutralized (time-series, per entity) and then fundamentally
# neutralized (cross-sectional, per date). Rows missing any
# fundamental value keep their (market-residualized) value
# unchanged rather than becoming NaN.
#
# DEFAULT: False — flip to True to run the fundamental-factor
# variant. Both branches use the same downstream code.
# =============================================================

RESIDUALIZE_VS_FUNDAMENTALS = False   # ← v18 default: fundamentals are hypothesis-carrying FEATURES

FUND_FACTOR_COLS = ["roe_lag1", "profit_margin_lag1",
                    "revenue_growth_lag1", "debt_to_equity_lag1"]

def residualize_cross_sectional(df, feature_col, factor_cols,
                                min_stocks=50):
    """
    Per-date cross-sectional residualization of feature_col
    against factor_cols. Returns a Series aligned to df.index.
    Rows with missing factors keep their original value.
    """
    out = df[feature_col].copy()
    n_dates_done = 0
    for date, idx in df.groupby("date").groups.items():
        sub = df.loc[idx, [feature_col] + factor_cols].dropna()
        if len(sub) < min_stocks:
            continue
        X = sm.add_constant(sub[factor_cols])
        y = sub[feature_col]
        try:
            model = sm.OLS(y, X).fit()
            out.loc[sub.index] = model.resid
            n_dates_done += 1
        except Exception:
            continue
    return out, n_dates_done

if RESIDUALIZE_VS_FUNDAMENTALS:
    print("=" * 60)
    print("Cell 12b: RESIDUALIZING STOCK FEATURES AGAINST FUNDAMENTALS")
    print("=" * 60)
    missing = [c for c in FUND_FACTOR_COLS if c not in final_df.columns]
    if missing:
        print(f"Cell 12b: WARNING — missing factor columns {missing}; skipping.")
    else:
        for col in STOCK_FEATURES_RAW:
            if col not in final_df.columns:
                continue
            resid, n_ok = residualize_cross_sectional(
                final_df, col, FUND_FACTOR_COLS
            )
            final_df[f"{col}_prefund"] = final_df[col]  # keep pre-step copy
            final_df[col] = resid
            print(f"  Cell 12b: {col:<22} residualized on {n_ok} dates")
        print("\nCell 12b: Done. Fundamental columns will be EXCLUDED from")
        print("Cell 12b: ALL_FEATURES in Cell 13 (they are now risk factors,")
        print("Cell 12b: not features). Pre-step values kept in *_prefund.")
else:
    print("Cell 12b: RESIDUALIZE_VS_FUNDAMENTALS = False — skipped.")
    print("Cell 12b: (Set the flag to True to neutralize stock features")
    print("Cell 12b:  against ROE / margin / revenue growth / D/E and drop")
    print("Cell 12b:  fundamentals from the model feature set.)")

In [ ]:
# =============================================================
# Cell 13 — Feature List + Data Quality Audit
# =============================================================
# Lock in the canonical feature set and run a thorough
# data quality audit BEFORE any modelling.
#
# FEATURE SETS:
# A) STOCK_FEATURES (7):      momentum, vol, ma_spread
#    → cross-sectionally z-scored (standardised per date)
#    → winsorised at 1%/99% per date
#
# B) MACRO_ETF_FEATURES (5):  yield_spread, spy/hyg/gold/oil
#    → NOT cross-sectionally standardised (market-wide signals)
#
# C) SHORT_VOL_FEATURES (3):  short_ratio, z20, chg5
#    → cross-sectionally z-scored (stock-specific signal)
#    → available from Jan 2021 onwards
#
# D) FUNDAMENTAL_FEATURES (4): roe, profit_margin,
#                               revenue_growth, debt_to_equity
#    → cross-sectionally z-scored
#    → available from Jan 2021 onwards
#
# E) SHORT_INT_FEATURES (2): days_to_cover, short_interest_rank
#    → COMPUTED in Cell 9 but EXCLUDED from ALL_FEATURES.
#    → REASON: only available from Dec 2022 onwards, covering
#      180 of 570+ tickers. Including it in the global dropna
#      (model_df construction in Cell 14) would discard all
#      2020-2022 training data for every OTHER feature too —
#      an unacceptable tradeoff given short interest's limited
#      incremental signal (see Cell 16 IC audit).
#    → This is a deliberate, documented research decision, not
#      an oversight. Short interest remains in final_df for
#      anyone who wants to build a Dec-2022-onwards-only model.
#
# OUTLIER TREATMENT:
# All stock-level features winsorised at 1%/99% cross-sectionally
# per date. This caps extreme values without dropping rows.
# Fundamental ratios need this especially (ROE, D/E can be extreme).
# =============================================================

print("Cell 13: Defining canonical feature list...")

STOCK_FEATURES = [
    "mom_5_lag1","mom_20_lag1","mom_60_lag1","mom_252_lag1",
    "vol_20_lag1","vol_60_lag1","ma_spread_lag1",
    "yz_vol_20_lag1",    # v17: Yang-Zhang range-based OHLC volatility
    "hurst_126_lag1",    # v17: rolling 6-month Hurst exponent
]
MACRO_ETF_FEATURES = [
    "yield_spread_lag1","spy_tlt_lag1","hyg_tlt_lag1",
    "gold_ret_lag1","oil_ret_lag1",
]
SHORT_VOL_FEATURES = [
    "short_ratio_lag1","short_ratio_z20_lag1","short_ratio_chg5_lag1",
]
FUNDAMENTAL_FEATURES = [
    "roe_lag1","profit_margin_lag1",
    "revenue_growth_lag1","debt_to_equity_lag1",
]

# NOTE: SHORT_INT_FEATURES is defined for reference/transparency
# but deliberately NOT included in ALL_FEATURES below.
SHORT_INT_FEATURES = [
    "days_to_cover_lag1","short_interest_rank_lag1",
]

# ── v18: MACRO/ETF FEATURES ARE EXCLUDED FROM THE MODEL ──
# They are identical for every stock on a given day, so they carry ZERO
# cross-sectional ranking information by construction. Including them
# with a raw-return target is what turned v17's Random Forest into a
# market timer with constant within-day predictions (the "IC=+0.12,
# N=1" artifact). They stay in the pipeline only as the market factor
# for residualization (Cell 12) and for regime context.
#
# v18: every modelled feature carries a directional hypothesis, stated
# BEFORE looking at results and checked against training IC in Cell 16.
FEATURE_HYPOTHESES = {
    "mom_5_lag1":            ("-", "short-term reversal: 1-week winners revert"),
    "mom_20_lag1":           ("+", "medium momentum persists"),
    "mom_60_lag1":           ("+", "medium momentum persists"),
    "mom_252_lag1":          ("+", "long momentum persists"),
    "vol_20_lag1":           ("-", "low-vol anomaly: high vol underperforms"),
    "vol_60_lag1":           ("-", "low-vol anomaly: high vol underperforms"),
    "yz_vol_20_lag1":        ("-", "low-vol anomaly (range-based estimate)"),
    "ma_spread_lag1":        ("-", "overextension above trend reverts"),
    "hurst_126_lag1":        ("+", "persistent (trending) names continue"),
    "short_ratio_lag1":      ("-", "short-selling pressure is informed"),
    "short_ratio_z20_lag1":  ("-", "unusual shorting spike is bearish"),
    "short_ratio_chg5_lag1": ("-", "rising shorting momentum is bearish"),
    "roe_lag1":              ("+", "quality premium"),
    "profit_margin_lag1":    ("+", "quality premium"),
    "revenue_growth_lag1":   ("+", "growth/earnings momentum"),
    "debt_to_equity_lag1":   ("-", "leverage penalised"),
}

if globals().get("RESIDUALIZE_VS_FUNDAMENTALS", False):
    # Fundamentals used as risk factors (Cell 12b) → not features.
    ALL_FEATURES = (STOCK_FEATURES + SHORT_VOL_FEATURES)
    print("Cell 13: Fundamental-residualization mode ACTIVE —")
    print("Cell 13: fundamentals EXCLUDED from ALL_FEATURES (risk factors).")
else:
    ALL_FEATURES = (STOCK_FEATURES + SHORT_VOL_FEATURES +
                    FUNDAMENTAL_FEATURES)
print("Cell 13: Macro/ETF features EXCLUDED from ALL_FEATURES (v18):")
print("Cell 13: constant within a day ⇒ no cross-sectional information.")

# Filter to only cols that exist
ALL_FEATURES = [f for f in ALL_FEATURES if f in final_df.columns]
print(f"Cell 13: Active features ({len(ALL_FEATURES)}): {ALL_FEATURES}")
print(f"Cell 13: Short interest features EXCLUDED from ALL_FEATURES")
print(f"Cell 13: (computed but not modelled — see decision note above)")

# ── Data quality audit ──
print("\n" + "="*60)
print("Cell 13: DATA QUALITY AUDIT")
print("="*60)

df_model = final_df.dropna(subset=["target"]).copy()
print(f"Cell 13: Rows with valid target: {len(df_model):,}")
print(f"Cell 13: Tickers: {df_model['ticker'].nunique()}")

print("\nCell 12: --- Feature missingness (% missing) ---")
miss = df_model[ALL_FEATURES].isna().mean() * 100
for f, v in miss.items():
    status = "✓" if v < 5 else ("⚠" if v < 40 else "✗")
    print(f"  {status} {f:<40} {v:.2f}%")

print("\nCell 12: --- Short interest missingness (excluded, shown for reference) ---")
si_miss = df_model[SHORT_INT_FEATURES].isna().mean() * 100
for f, v in si_miss.items():
    print(f"  ⓘ {f:<40} {v:.2f}%  (excluded from model)")

# ── Outlier check on a sample date ──
print("\nCell 12: --- Outlier check (sample cross-section) ---")
sample_date = df_model["date"].median()
sample = df_model[df_model["date"] == sample_date]
check_cols = STOCK_FEATURES + SHORT_VOL_FEATURES[:1] + FUNDAMENTAL_FEATURES[:2]
for f in check_cols:
    if f in sample.columns:
        s = sample[f].dropna()
        if len(s) > 0:
            print(f"  {f:<40} mean={s.mean():.4f}  std={s.std():.4f}  "
                  f"min={s.min():.4f}  max={s.max():.4f}")

# ── Winsorise ALL modelled stock-level lag features ──
print("\nCell 12: Winsorising at 1%/99% per date...")
COLS_TO_WINSORISE = (STOCK_FEATURES + SHORT_VOL_FEATURES + FUNDAMENTAL_FEATURES)

for col in COLS_TO_WINSORISE:
    if col not in final_df.columns:
        continue
    lo = final_df.groupby("date")[col].transform(lambda x: x.quantile(0.01))
    hi = final_df.groupby("date")[col].transform(lambda x: x.quantile(0.99))
    final_df[col] = final_df[col].clip(lower=lo, upper=hi)

# ── Cross-sectional z-score ALL modelled stock-level lag features ──
print("Cell 13: Cross-sectionally z-scoring stock/short-vol/fundamental features...")
COLS_TO_ZSCORE = COLS_TO_WINSORISE  # same set

for col in COLS_TO_ZSCORE:
    if col not in final_df.columns:
        continue
    final_df[col] = final_df.groupby("date")[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-9)
    )

print("Cell 13: Data quality audit and cleaning complete.")

In [ ]:
# =============================================================
# Create model_df  (the modelling panel)
# =============================================================
# Drop rows without a full feature vector or a valid target. This is
# the canonical panel every downstream stage trains and trades on.
# (Originally the top of Cell 14; the train/val/test split now lives
#  in notebook 2.)
# =============================================================

model_df = final_df.dropna(subset=ALL_FEATURES + ["target"]).copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna(subset=ALL_FEATURES + ["target"])
model_df = model_df.sort_values(["date", "ticker"]).reset_index(drop=True)

print(f"model_df: {model_df.shape[0]:,} rows x {model_df.shape[1]} cols")
print(f"  dates:   {model_df['date'].min().date()} -> {model_df['date'].max().date()}")
print(f"  tickers: {model_df['ticker'].nunique()}  |  features: {len(ALL_FEATURES)}")


In [ ]:
# =============================================================
# Persist model_df + pipeline metadata for the downstream notebooks
# =============================================================
import json
os.makedirs("Data/interim", exist_ok=True)

model_df.to_parquet("Data/interim/model_df.parquet", index=False)

meta = {
    "START_DATE":           START_DATE,
    "TARGET_HORIZON":       TARGET_HORIZON,
    "REBALANCE_EVERY":      REBALANCE_EVERY,
    "PERIODS_PER_YEAR":     PERIODS_PER_YEAR,
    "ALL_FEATURES":         ALL_FEATURES,
    "STOCK_FEATURES":       STOCK_FEATURES,
    "MACRO_ETF_FEATURES":   MACRO_ETF_FEATURES,
    "SHORT_VOL_FEATURES":   SHORT_VOL_FEATURES,
    "FUNDAMENTAL_FEATURES": FUNDAMENTAL_FEATURES,
    "SHORT_INT_FEATURES":   SHORT_INT_FEATURES,
    "FEATURE_HYPOTHESES":   {k: list(v) for k, v in FEATURE_HYPOTHESES.items()},
}
with open("Data/interim/pipeline_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved -> Data/interim/model_df.parquet", model_df.shape)
print("Saved -> Data/interim/pipeline_meta.json")
